# Exam Project - Introduction to Social Data Science
August 28, 2024

## Project: Forecasting Vote Counts for Danish Borgerforslag

## Group 7:
- Oliver Nyrop Weeks (vsn684)
- Sofus Galavits Møller (qvc730)
- Victor V. Kristensen (gcp458)
- Jonas T. Schmidt (mcp656)

In [1]:
# import modules
import pandas as pd                                                         # We use pandas for datahandling 
import numpy as np
import Stemmer                                                              # For stemming in Danish
import lemmy                                                                # For lemmatization

# import classes
from afinn import Afinn
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords                                           # For stopwords in Danish
from sklearn.feature_extraction.text import CountVectorizer                 # Used for topic tagging with LDA
from sklearn.decomposition import LatentDirichletAllocation                 # Used for topic tagging with LDA

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/olivernyropweeks/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Load the dataset:

In [2]:
# Load the dataset from the CSV file
loaded_df = pd.read_csv("data/merged_df_output.csv")

# Display the first few rows of the loaded DataFrame
loaded_df

,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,...,lifetime,Start Month,Combined Text,Title Length,Combined Text Length,Title Uppercase Count,Combined Text Uppercase Count,Title Lowercase Count,Combined Text Lowercase Count,Combined Text LIX Score
0,Afskaf inklusionsloven,2024-08-15,2025-02-11,39.000,FT-18153,For at forbedre undervisningen og sikre optima...,"Inklusionsloven, som blev indført i 2012 med d...",3,Aarhus,Aarhus,...,3,8,For at forbedre undervisningen og sikre optima...,22,3102,1,18,20,2556,59.349155
1,Fartbøder forhindrer i at opnå Dansk statsborg...,2024-08-06,2025-02-02,82.000,FT-18099,Forslaget drejer sig om regler for fartbøder v...,Vi synes ikke det er rimeligt at en fartbøde s...,3,Vejle,Vejle,...,12,8,Forslaget drejer sig om regler for fartbøder v...,52,1283,2,17,44,999,40.803930
2,Forbyd dressurridning som konkurrencesport,2024-08-06,2025-02-02,245.000,FT-18075,Dressurridning som konkurrencesport er en spor...,Dette borgerforslag er stillet at dyreetiske å...,3,Hvidovre,Adressebeskyttelse,...,12,8,Dressurridning som konkurrencesport er en spor...,42,2326,1,40,38,1864,40.113388
3,Lavere skat og fjernelse af minimumsalder på p...,2024-08-06,2025-02-02,338.000,FT-18046,Vi stiller et forslag om ændring af reglerne f...,Vi stiller dette forslag for at lette den økon...,3,Randers,København,...,12,8,Vi stiller et forslag om ændring af reglerne f...,100,1764,1,14,87,1429,50.907995
4,Bloddonorpligt. Som værnepligt,2024-07-26,2025-01-22,41.000,FT-18044,Jeg forslår at der etableres en pligt til at a...,Det er jo velkendt at danske regioner mangler ...,3,Hjørring,Hjørring,...,23,7,Jeg forslår at der etableres en pligt til at a...,30,702,2,11,25,544,30.552941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,2018-01-30,2018-07-29,189.000,FT-00139,Jeg stiller hermed forslag om at ophævelsen af...,"Af hensyn til forurening.\nAf hensyn til at ""4...",3,Odsherred,Odsherred,...,180,1,Jeg stiller hermed forslag om at ophævelsen af...,41,545,1,6,33,426,43.400000
1843,Automatisk førtidspension til personer der har...,2018-01-30,2018-07-29,66.000,FT-00059,Forslaget er at man automatisk giver førtidspe...,Formålet med lovforslaget er at sikre ofrene f...,4,Vejen,Sønderborg,...,180,1,Forslaget er at man automatisk giver førtidspe...,125,1147,1,6,108,944,53.763418
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,2018-01-30,2018-07-29,597.000,FT-00034,"Lovforslaget går i sin enkelhed ud på, at sikr...",dette er blot en ud af alt for mange anbragte ...,3,Kalundborg,Kalundborg,...,180,1,"Lovforslaget går i sin enkelhed ud på, at sikr...",102,1801,88,13,0,1385,33.292443
1845,"Statsborgerskab til unge mennesker, som er fød...",2018-01-30,2018-07-29,6.263,FT-00067,Folketinget pålægger regeringen at genindføre ...,"Mennesker som er født og opvokset i Danmark, o...",3,Stevns,København,...,180,1,Folketinget pålægger regeringen at genindføre ...,105,4863,2,46,88,3886,49.983954


In [3]:
# Danish stopwords. This is a small list, enough?
stop = stopwords.words('danish')

# Stemmer in Danish
stemmer = Stemmer.Stemmer('danish')

# Lemmatizer in Danish
lemmatizer = lemmy.load("da")

# Create an empty list to store the results
body_tokens = []
body_tokens_stem = []
body_tokens_lem = []

# Loop through each title in the DataFrame column "title"
for document in loaded_df["Combined Text"]:

    # Split the title into individual words
    tokens = document.split()

    # Removing stopwords to focus on more meaningful words
    tokens_stop = [i for i in tokens if i not in stop]

    # Stemming: Basicly removing suffixes
    tokens_stemmed = stemmer.stemWords(tokens_stop)

    # Apply lemmatization
    tokens_lem = [lemmatizer.lemmatize("", token)[0].lower() for token in tokens_stop]  # Assuming the first lemma is desired

    # Convert the lists of tokens back to a single string (sentence)
    sentence_stemmed = " ".join(tokens_stemmed)
    sentence_lemmatized = " ".join(tokens_lem)
    sentence_tokens = " ".join(tokens_stop)

    # Append the results
    body_tokens_stem.append(sentence_stemmed)
    body_tokens_lem.append(sentence_lemmatized)
    body_tokens.append(sentence_tokens)

# Print the first few rows to verify
print("Original Tokens:", body_tokens[0])
print("Stemmed Tokens:", body_tokens_stem[0])
print("Lemmatized Tokens:", body_tokens_lem[0])


Original Tokens: For forbedre undervisningen sikre optimal støtte elever særlige behov, foreslås afskaffe inklusionsloven stedet genindføre styrke specialtilbuddene folkeskolen. Dette indebærer omlægning ressourcerne, så flere midler afsættes specialskoler specialklasser, elever særlige behov kan få skræddersyet undervisning støtte uddannede specialister. Der desuden etableres flere fleksible ordninger, elever kan modtage kombination specialpædagogisk støtte undervisning almindelige klasser, hensigtsmæssigt udvikling. Dette sikrer, får nødvendige faglige sociale støtte, samtidig højere grad inkluderes skolens fællesskab præmisser. Lærernes kompetencer bør opgraderes gennem efteruddannelse, så bedre rustet identificere håndtere elever særlige behov. Desuden bør etableres stærkere samarbejdsstrukturer mellem lærere, pædagoger specialister sikre helhedsorienteret tilgang hver enkelt elevs behov. Denne ændring sikre, elever modtager undervisning støtte niveau, bedst fremmer læring trivsel,

In [4]:
# Set min_df to 3 (meaning that terms apearing 3 times or less will be removed). We set max_df to 0.1 (meaning that terms apearing in 10% or more documents will be removed)
count = CountVectorizer(min_df=3, max_df=0.1, max_features=50000)
bag = count.fit_transform(body_tokens_lem) # Fit our bag-of-words (given above specifications) and form a bag.

# Define the unsupervised machine learning model with varying components
lda = LatentDirichletAllocation(n_components=10, random_state=123)
borgerforslag_topics = lda.fit_transform(bag) # Borgerforslag_topics contain the topic distribution of each document: Each row represents a document, and each column represents a topic. The values in this matrix represent the probability that a given topic contributes to the document.

n_top_words = 10
word_names = count.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_): #lda.components_ stores a matrix containing the word importance for each topic
    print("Topic %d:" % (topic_idx + 1))
    print(" ".join([word_names[i]
    for i in topic.argsort()\
        [:-n_top_words - 1:-1]]))
    

Topic 1:
cannabis israel gaza and støje of vindmølle folkedrab db international
Topic 2:
hund fyrværkeri syg patient sygdom behandling læge dy dyr vurdering
Topic 3:
parti co2 valg natur politiker politisk miljø folketinget stemme adgang
Topic 4:
elev undervisning følelse folkeskole lære klasse udgift elevere skole skat
Topic 5:
køre grøn bile trafik produktion landbrug reducere ukraine kilometer verden
Topic 6:
seksuel straffe overgreb mand politi risiko beskytte 18 2020 misbruge
Topic 7:
virksomhed uge barsel procent studerende digital aktiv eller who måned
Topic 8:
socialrådgiver aarhus undervisning eksamen elevere container havn autorisation anbringe statsborgerskab
Topic 9:
politisk ansvar myndighed befolkning køn politi demokratisk rettighed adgang enhver
Topic 10:
kvinde uddannelse su eu mand studerende sygdom psykisk behandling kvinder


In [9]:
import numpy as np
import pandas as pd

# Assuming borgerforslag_topics is an array with the topic distributions for each document
# and loaded_df is the DataFrame that has been loaded with the dataset

# Assign the most relevant topics to each document
doc_labels = []
for i, topic_dist in enumerate(borgerforslag_topics):
    # Sort topics by probability in descending order
    sorted_topics = np.argsort(topic_dist)[::-1] + 1  # +1 to make topic index human-readable
    
    # Determine the number of relevant topics based on a threshold or top N selection
    top_n = 3  # For example, we want to assign up to 3 topics
    top_topics = sorted_topics[:top_n]
    
    # Filter topics based on a probability threshold (optional)
    threshold = 0.35  # Only include topics with a probability above 0.35
    relevant_topics = [topic for topic in top_topics if topic_dist[topic - 1] > threshold]
    
    # Assign the relevant topics to the document
    for topic in relevant_topics:
        doc_labels.append((i, topic))

# Create a DataFrame from the doc_labels with the index as document index
doc_labels_df = pd.DataFrame(doc_labels, columns=['index', 'topic'])

# Create dummies for the topics and sum them by document index to avoid duplicates
topic_dummies = pd.get_dummies(doc_labels_df['topic'], prefix="topic")
topic_dummies = doc_labels_df.join(topic_dummies).groupby('index').sum().reset_index(drop=True)

# Concatenate the topic dummies with the original DataFrame
df_xy = pd.concat([loaded_df, topic_dummies], axis=1)

# Display the resulting DataFrame
df_xy


,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,...,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10
0,Afskaf inklusionsloven,2024-08-15,2025-02-11,39.000,FT-18153,For at forbedre undervisningen og sikre optima...,"Inklusionsloven, som blev indført i 2012 med d...",3,Aarhus,Aarhus,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,Fartbøder forhindrer i at opnå Dansk statsborg...,2024-08-06,2025-02-02,82.000,FT-18099,Forslaget drejer sig om regler for fartbøder v...,Vi synes ikke det er rimeligt at en fartbøde s...,3,Vejle,Vejle,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,Forbyd dressurridning som konkurrencesport,2024-08-06,2025-02-02,245.000,FT-18075,Dressurridning som konkurrencesport er en spor...,Dette borgerforslag er stillet at dyreetiske å...,3,Hvidovre,Adressebeskyttelse,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Lavere skat og fjernelse af minimumsalder på p...,2024-08-06,2025-02-02,338.000,FT-18046,Vi stiller et forslag om ændring af reglerne f...,Vi stiller dette forslag for at lette den økon...,3,Randers,København,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,Bloddonorpligt. Som værnepligt,2024-07-26,2025-01-22,41.000,FT-18044,Jeg forslår at der etableres en pligt til at a...,Det er jo velkendt at danske regioner mangler ...,3,Hjørring,Hjørring,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,2018-01-30,2018-07-29,189.000,FT-00139,Jeg stiller hermed forslag om at ophævelsen af...,"Af hensyn til forurening.\nAf hensyn til at ""4...",3,Odsherred,Odsherred,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1843,Automatisk førtidspension til personer der har...,2018-01-30,2018-07-29,66.000,FT-00059,Forslaget er at man automatisk giver førtidspe...,Formålet med lovforslaget er at sikre ofrene f...,4,Vejen,Sønderborg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,2018-01-30,2018-07-29,597.000,FT-00034,"Lovforslaget går i sin enkelhed ud på, at sikr...",dette er blot en ud af alt for mange anbragte ...,3,Kalundborg,Kalundborg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1845,"Statsborgerskab til unge mennesker, som er fød...",2018-01-30,2018-07-29,6.263,FT-00067,Folketinget pålægger regeringen at genindføre ...,"Mennesker som er født og opvokset i Danmark, o...",3,Stevns,København,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
len(doc_labels)

2151

In [7]:
topic_dummies

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10
0,False,False,False,False,False,False,False,True,False,False
1,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,True,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False
4,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
2146,False,True,False,False,False,False,False,False,False,False
2147,False,False,False,False,False,True,False,False,False,False
2148,False,False,False,False,False,False,False,False,False,True
2149,False,False,False,False,False,False,False,False,True,False
